<a href="https://colab.research.google.com/github/hjmohamedbilal/movies-dataset/blob/main/MohamedBilalHJLabProgram2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab Program 2: Data Cleaning on an Open-Source Dataset

Step 1: Import the required libraries

In [2]:
!python -m pip install matplotlib seaborn pandas numpy

In [3]:
import pandas as pd
import numpy as np
import os

Step 2: Load the Dataset

In [8]:
# Manually inspect the raw lines around row 561
with open('zomato.csv', 'r', encoding='latin-1') as f:
    lines = f.readlines()

# Print lines 550 to 570 (0-indexed indices 549 to 569)
for idx in range(549, min(569, len(lines))):
    print(f"Line {idx + 1}: {lines[idx].strip()}\n" + "-"*40)

Line 550: https://www.zomato.com/bangalore/great-indian-rolls-basavanagudi-bangalore?context=eyJzZSI6eyJlIjpbIjE4NjQ2NDg1IiwiMTg3MjkwMTQiLCIxODcyOTQwOSIsIjE4NzEzOTk1IiwxODcyNzIwMywiMTg3MjgwNDUiLCIxODcyNzI3NSIsIjE4NzI3Nzk1IiwiMTg3MjgxNjciLCIxODcyMzE3MyIsIjE4Njk4NjQ2IiwiMTg3MDI1MTciLCIxODY4OTE4NCIsIjE4NzA3MjA3IiwiMTc5NTY3MDUiLCI1MDI4MyIsIjUwNDEzIiwiNTE0MDMiLCI1MDg0MyIsIjUxMDk1Il0sInQiOiJEZWxpdmVyeSBSZXN0YXVyYW50cyBpbiBCYW5hc2hhbmthcmkifX0=,"Near Metro Station R.V.Road, Basavanagudi, Bangalore",Great Indian Rolls,Yes,No,3.3/5,4,"+91 9686631631
----------------------------------------
Line 551: +91 9740088890",Basavanagudi,Quick Bites,,Rolls,100,"[('Rated 3.0', 'RATED\n  It located behind the SOUTH END metro station a small place with no seating . The cook were polite and polite . We had chicken rolls which was amazing . The chicken was well marinated and was a perfect roll .\nWe also tries the VEG CHEESY PASTA was not that good as compared to other white sauce pasta . It was a cause for m

In [10]:
# Load the dataset. latin-1 encoding handles special characters.
# To prevent unclosed quote errors (like row 561 EOF error), we set quoting=3 (csv.QUOTE_NONE).
import csv
df = pd.read_csv("zomato.csv", encoding="latin-1", quoting=csv.QUOTE_NONE, on_bad_lines="skip")
print("Zomato Bangalore dataset loaded successfully.")
print("Initial shape (rows, columns):", df.shape)
print("-" * 70)
print("First 5 rows of the dataset:")
print(df.head())

Zomato Bangalore dataset loaded successfully.
Initial shape (rows, columns): (278, 17)
----------------------------------------------------------------------
First 5 rows of the dataset:
                                                 url      address  \
0  https://www.zomato.com/bangalore/jalsa-banasha...         "942   
1  https://www.zomato.com/bangalore/grand-village...          "10   
2  https://www.zomato.com/bangalore/timepass-dinn...          "37   
3  https://www.zomato.com/bangalore/rosewood-inte...        "19/1   
4                                      080 26740366"  Mysore Road   

                     name                online_order  \
0          21st Main Road                   2nd Stage   
1               3rd Floor          Lakshmi Associates   
2                     5-1                   4th Floor   
3   New Timberyard Layout   Beside Satellite Bus Stop   
4           Casual Dining                         NaN   

                 book_table                      rate  

Step 3: Inspect the dataset

In [11]:
print("Dataset information:")
df.info()
print("\nMissing values before cleaning:")
print(df.isnull().sum())
print("-" * 70)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278 entries, 0 to 277
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   url                          278 non-null    object
 1   address                      278 non-null    object
 2   name                         278 non-null    object
 3   online_order                 239 non-null    object
 4   book_table                   274 non-null    object
 5   rate                         278 non-null    object
 6   votes                        278 non-null    object
 7   phone                        278 non-null    object
 8   location                     278 non-null    object
 9   rest_type                    274 non-null    object
 10  dish_liked                   263 non-null    object
 11  cuisines                     239 non-null    object
 12  approx_cost(for two people)  153 non-null    object
 13  reviews_list  

Step 4:Define the cleaning functions

Function 1: Handle missing values(mean/mode imputation)

In [12]:
def clean_missing(data):
  """Fill missing values: numeric columns with the mean, categorical
  columns with the mode (most frequent value). Returns the cleaned
  DataFrame."""
  print("\nCleaning missing values...")
  # --- Numeric columns: fill missing values with the column mean ---
  numeric_columns = data.select_dtypes(include=np.number).columns
  for col in numeric_columns:
    missing_count = data[col].isnull().sum()
    if missing_count > 0:
      mean_value = data[col].mean()
      data[col].fillna(mean_value, inplace=True)
      print(f"Filled {missing_count} missing values "
      f"in '{col}' using MEAN: {mean_value:.2f}")
  # --- Categorical columns: fill missing values with the mode ---
  categorical_columns = data.select_dtypes(include="object").columns
  for col in categorical_columns:
    missing_count = data[col].isnull().sum()
    if missing_count > 0:
      mode_value = data[col].mode()[0] # most frequent value
      data[col] = data[col].fillna(mode_value, inplace=True)
      print(f"Filled {missing_count} missing values "
            f"in '{col}' using MODE: {mode_value}")
      print("\nMissing value handling completed.")
      return data


Function 2: Remove duplicate rows

In [13]:
def remove_duplicates(data):
  """Remove exact duplicate rows from the DataFrame and report how many
  were found and removed."""
  print("\nChecking for duplicate rows...")
  rows_before = data.shape[0]
  duplicate_count = data.duplicated().sum()
  print("Duplicate rows found:", duplicate_count)
  data = data.drop_duplicates()
  rows_after = data.shape[0]
  print("Rows before removing duplicates:", rows_before)
  print("Rows after removing duplicates:", rows_after)
  print("Duplicate removal completed.")
  return data

Function 3: One-hot encoding for categorical columns

In [14]:
def encode_categorical(data):
  """Convert all categorical (text) columns to numeric columns using
  one-hot encoding. Returns a new DataFrame with the encoded columns."""
  print("\nConverting categorical columns to numeric (one-hot encoding)...")
  categorical_columns = data.select_dtypes(include="object").columns
  print("Categorical columns:", list(categorical_columns))
  # pd.get_dummies creates one binary column per category value.
  # drop_first=True avoids redundant (perfectly correlated) columns.
  data_encoded = pd.get_dummies(
      data,
      columns=categorical_columns,
      drop_first=True
  )
  print("One-hot encoding completed.")
  print("Number of categorical columns encoded:", len(categorical_columns))
  return data_encoded

Step 5: Run the cleaning pipeline

In [15]:
print("=" * 70)
print("BEFORE CLEANING")
print("=" * 70)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

BEFORE CLEANING
Rows: 278
Columns: 17


In [16]:
#1. Handle missing values
df = clean_missing(df)


Cleaning missing values...
Filled 39 missing values in 'online_order' using MODE:  24th Cross

Missing value handling completed.


/tmp/ipykernel_512/716858328.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col] = data[col].fillna(mode_value, inplace=True)


In [17]:
# 2. Remove duplicate rows
df = remove_duplicates(df)


Checking for duplicate rows...
Duplicate rows found: 0
Rows before removing duplicates: 278
Rows after removing duplicates: 278
Duplicate removal completed.


In [18]:
# 3. One-hot encode categorical columns
# Note: this expands the number of columns, since each category value
# becomes its own 0/1 column.
df_clean = encode_categorical(df)


Converting categorical columns to numeric (one-hot encoding)...
Categorical columns: ['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes', 'phone', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_cost(for two people)', 'reviews_list', 'menu_item', 'listed_in(type)', 'listed_in(city)']
One-hot encoding completed.
Number of categorical columns encoded: 17


Step 6: Inspect the results after cleaning

In [19]:
print("=" * 70)
print("AFTER CLEANING")
print("=" * 70)
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])
print("-" * 70)
print("First 5 rows of the dataset:")
print(df_clean.head())

AFTER CLEANING
Rows: 278
Columns: 1624
----------------------------------------------------------------------
First 5 rows of the dataset:
   url_+91 7019192360"  url_+91 7022151183"  url_+91 7204025617"  \
0                False                False                False   
1                False                False                False   
2                False                False                False   
3                False                False                False   
4                False                False                False   

   url_+91 7204756477"  url_+91 7411148111"  url_+91 7752956789"  \
0                False                False                False   
1                False                False                False   
2                False                False                False   
3                False                False                False   
4                False                False                False   

   url_+91 7760539140"  url_+91 7892206383"

Step 7: Verify that no missing values remain

In [20]:
print("Total missing values remaining:", df.isnull().sum().sum())

Total missing values remaining: 1446


Step 8: Generate a data quality report

In [21]:
print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)
report = pd.DataFrame({
    "Column_Name": df.columns,
    "Null_Count": df.isnull().sum().values,
    "Unique_Values": df.nunique().values
    })
print(report.head())
report.to_csv("data_quality_report.csv", index=False)
print("\nData quality report saved as: data_quality_report.csv")


DATA QUALITY REPORT
    Column_Name  Null_Count  Unique_Values
0           url           0            273
1       address           0            169
2          name           0            160
3  online_order         278              0
4    book_table           4            120

Data quality report saved as: data_quality_report.csv


Step 9: Save the cleaned dataset


In [22]:
output_file = "zomato_bangalore_cleaned.csv"
df_clean.to_csv(output_file, index=False)
print("Cleaned dataset saved as:", output_file)

Cleaned dataset saved as: zomato_bangalore_cleaned.csv


Step 10: Summary
A quick recap of everything this notebook did to the dataset.

In [23]:
print("=" * 70)
print("DATA CLEANING COMPLETED SUCCESSFULLY")
print("=" * 70)
print("\nOperations performed:")
print("1. Missing values handled using mean (numeric) and mode (categorical)")
print("2. Duplicate rows removed")
print("3. Categorical columns converted using one-hot encoding")
print("4. Data quality report generated")
print("5. Cleaned dataset saved to disk")

DATA CLEANING COMPLETED SUCCESSFULLY

Operations performed:
1. Missing values handled using mean (numeric) and mode (categorical)
2. Duplicate rows removed
3. Categorical columns converted using one-hot encoding
4. Data quality report generated
5. Cleaned dataset saved to disk


Step 11: View the saved data quality report

In [24]:
report = pd.read_csv("data_quality_report.csv")
display(report)
print("Report saved at:", os.path.abspath("data_quality_report.csv"))

,Column_Name,Null_Count,Unique_Values
0,url,0,273
1,address,0,169
2,name,0,160
3,online_order,278,0
4,book_table,4,120
5,rate,0,90
6,votes,0,108
7,phone,0,100
8,location,0,83
9,rest_type,4,73


Report saved at: /content/data_quality_report.csv
